In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import seaborn as sns
import os 
from pathlib import Path

from utils import *


import warnings
warnings.filterwarnings("ignore")

In [ ]:
# ----------------------------------------
# Generate references (r_x, r_y)
# ----------------------------------------
def generate_sum_of_sinusoids(t, amps, freqs, phases):
    r = np.zeros_like(t)
    for a, f, phi in zip(amps, freqs, phases):
        r += a * np.cos(2 * np.pi * f * t + phi)
    return r

def generate_sum_of_sinusoids_batch(t, amps, freqs, phases):
    all_r = np.zeros((phases.shape[0],len(t)))
    for i,cur_phases in enumerate(phases):
        r = np.zeros_like(t)
        for a, f, phi in zip(amps, freqs, cur_phases):
            r += a * np.cos(2 * np.pi * f * t + phi)
        all_r[i] = r
    return all_r

def ramp_function(t, ramp_duration=5.0):
    return np.clip(t / ramp_duration, 0, 1)

total_time = 30.0    # total time
sampling_rate = 10 # 10   # Hz
n_samples = int(total_time * sampling_rate)

t = np.linspace(0, total_time, n_samples, endpoint=False)

# For the x-axis
amps_x = np.array([2.31, 2.31,])
freqs_x = np.array([0.10,  0.30,])

# For the y-axis
amps_y = np.array([2.31, 2.31,])
freqs_y = np.array([0.20, 0.40,])

# ---------------------------------------------
# Instantiate RNN controller and the plant
# ---------------------------------------------
dt = 0.1
g  = 0.2
rnn = ct_rnn_controller(
    input_size=4,
    hidden_size=100,  
    output_size=2,
    phi='linear',
    manually_initialize=True,
    with_bias=False,
    train_Wih=False,     
    train_Whh=True,      
    train_Who=True,      
    small_scale=False,   
    Wih_start_big=False, 
    g=g, 
    dt=dt, 
    tracking_task=True,
)

plant = plant_2D_torch(dt=dt, noise=0.00, clamp=True)
params_to_train = [p for p in rnn.parameters() if p.requires_grad]
optimizer = optim.Adam(params_to_train, lr=1e-3)
w_grad_clip = 0


In [ ]:
# ------------------------------
# Training Loop
# ------------------------------
num_epochs = 5_001
batch_size = 100
train_losses = []

save_model = 1
net_name = 'closed_4_freq_linear_adam'
model_dir = Path(f'models/{net_name}')

if save_model:
    model_dir.mkdir(parents=True, exist_ok=True)

for epoch in range(num_epochs):

    # Initialize hidden state and system state
    h = torch.zeros((batch_size, rnn.hidden_size), dtype=torch.float32, requires_grad=True)
    x_current = torch.zeros((batch_size, 4), dtype=torch.float32, requires_grad=True)  # [x, vx, y, vy]

    # Generate random phases and sinusoidal reference
    phases_x = np.random.uniform(-np.pi, np.pi, size=(batch_size, len(amps_x)))
    phases_y = np.random.uniform(-np.pi, np.pi, size=(batch_size, len(amps_y)))

    r_x_full = generate_sum_of_sinusoids_batch(t, amps_x, freqs_x, phases_x)
    r_y_full = generate_sum_of_sinusoids_batch(t, amps_y, freqs_y, phases_y)

    ramp = ramp_function(t, ramp_duration=1.0)
    r_x = r_x_full * ramp
    r_y = r_y_full * ramp

    r_x_torch = torch.tensor(r_x, dtype=torch.float32)
    r_y_torch = torch.tensor(r_y, dtype=torch.float32)

    # Rollout and accumulate loss
    total_loss = 0.0
    for i in range(n_samples):
        input_rnn = torch.stack((x_current[:, 0], x_current[:, 2], r_x_torch[:, i], r_y_torch[:, i]), dim=1)
        u, h, _ = rnn(input_rnn, h)
        x_next = plant.step(x_current.T, u.T)

        mse_x = torch.mean((x_next[0, :] - r_x_torch[:, i]) ** 2)
        mse_y = torch.mean((x_next[2, :] - r_y_torch[:, i]) ** 2)
        total_loss += (mse_x + mse_y) * dt

        x_current = x_next.T

    loss = total_loss / n_samples

    # Backward and optimize
    if epoch > 0:
        optimizer.zero_grad()
        loss.backward()
        if w_grad_clip:
            torch.nn.utils.clip_grad_norm_(rnn.parameters(), max_norm=1.0)
        optimizer.step()

    train_losses.append(loss.item())

    if (epoch + 1) % 500 == 0:
        print(f"[Epoch {epoch + 1}/{num_epochs}] Loss = {loss.item():.4f}")

    # Save model checkpoint
    if save_model:
        torch.save({
            'epoch': epoch,
            'model_state': rnn.state_dict(),
            'loss': loss.item()
        }, model_dir / f'epoch_{epoch}.pth')


In [ ]:
# ------------------------------
# Load Training Losses
# ------------------------------
num_epochs = 5_000
net_name = 'closed_4_freq_linear_adam'
epochs = np.arange(0, num_epochs)
losses = []

for ep in epochs:
    checkpoint = torch.load(f'models/{net_name}/epoch_{ep}.pth')
    losses.append(checkpoint['loss'])

losses = np.array(losses)

# ------------------------------
# Plot Training Loss
# ------------------------------
plt.figure(figsize=(5, 4))
plt.plot(np.log(losses), label='Training loss', color='royalblue', lw=3)

plt.xlabel('Epoch')
plt.ylabel('Log Loss')
plt.title('Training Loss')
plt.legend()

# Mark selected epochs
highlight_epochs = [0,200,700,1800,4500]
for ep in highlight_epochs:
    plt.axvline(x=ep, ls='--', color='k', lw=1)

sns.despine()
plt.tight_layout()
plt.show()

phases_x = np.random.uniform(-np.pi, np.pi, size=len(amps_x))
phases_y = np.random.uniform(-np.pi, np.pi, size=len(amps_y))

# Define 4 stimulus conditions: single x or y frequency at a time
configs = [
    {"a_x": amps_x[:1], "f_x": freqs_x[:1], "a_y": np.zeros(1), "f_y": np.zeros(1)},
    {"a_x": np.zeros(1), "f_x": np.zeros(1), "a_y": amps_y[:1], "f_y": freqs_y[:1]},
    {"a_x": amps_x[1:], "f_x": freqs_x[1:], "a_y": np.zeros(1), "f_y": np.zeros(1)},
    {"a_x": np.zeros(1), "f_x": np.zeros(1), "a_y": amps_y[1:], "f_y": freqs_y[1:]}
]

colors = {'ref': sns.color_palette('deep', 4)[0], 'rnn': 'k'}
net_name = 'closed_4_freq_linear_adam'

for ep in highlight_epochs:
    fig, axes = plt.subplots(1, 6, figsize=(16, 2.5))
    ax0, ax1, ax2, ax3, ax4, ax5 = axes

    # Load model at this epoch
    rnn.load_state_dict(torch.load(f'models/{net_name}/epoch_{ep}.pth')['model_state'])

    for i, config in enumerate(configs):
        # Generate references
        r_x_full = generate_sum_of_sinusoids(t, config["a_x"], config["f_x"], phases_x)
        r_y_full = generate_sum_of_sinusoids(t, config["a_y"], config["f_y"], phases_y)
        ramp = ramp_function(t, ramp_duration=1.0)
        r_x = r_x_full * ramp
        r_y = r_y_full * ramp
        r_x_torch = torch.tensor(r_x, dtype=torch.float32)
        r_y_torch = torch.tensor(r_y, dtype=torch.float32)

        # Rollout trajectory
        with torch.no_grad():
            h = torch.zeros((1, rnn.hidden_size), dtype=torch.float32)
            x = torch.zeros(4, dtype=torch.float32)
            traj_x, traj_y = [], []

            for n in range(n_samples):
                input_t = torch.tensor([[x[0], x[2], r_x_torch[n], r_y_torch[n]]], dtype=torch.float32)
                u, h, _ = rnn(input_t, h)
                x = plant.step(x, u[0])
                traj_x.append(x[0].item())
                traj_y.append(x[2].item())

        # Plotting
        if i == 0:
            ax0.set_title(f'Ep: {ep}')
            ax0.plot(losses, color='k', lw=3)
            ax0.set_yscale('log')
            ax0.set_xlabel('Epoch')
            ax0.set_ylabel('Loss')
            ax0.axvline(x=ep, ls='--', color=colors['ref'])

            eigvals, _ = np.linalg.eig(-np.eye(100) + rnn.Whh.detach().numpy())
            ax1.scatter(eigvals.real, eigvals.imag, s=10)
            theta = np.linspace(0, 2 * np.pi, 200)
            ax1.plot(np.cos(theta) - 1, np.sin(theta), '--', color='black', lw=1.5, alpha=0.8)
            ax1.axvline(1, ls='--', color='k')
            ax1.axhline(0, color='black', lw=1.5)
            ax1.axvline(0, color='black', lw=1.5)
            ax1.set_xlim(-3, 3)
            ax1.set_ylim(-3, 3)
            ax1.set_xlabel('Real')
            ax1.set_ylabel('Imag')
            for omega in np.concatenate((freqs_x * 2 * np.pi, freqs_y * 2 * np.pi)):
                ax1.axhline(omega, color='k', alpha=0.5, lw=0.5)
                ax1.text(-2.5, omega, r'$\omega=$' + f'{omega:.2f}', size=8)

        # Select correct axis for each case
        axis_map = {0: ax2, 1: ax3, 2: ax4, 3: ax5}
        ax = axis_map[i]
        if i in [0, 2]:  # x trajectory
            ax.plot(t, r_x, color=colors['ref'], ls='-', label='Ref.')
            ax.plot(t, traj_x, color=colors['rnn'], ls='--', label='RNN')
            ax.set_ylabel("X")
            omega = config['f_x'][0] * 2 * np.pi
        else:  # y trajectory
            ax.plot(t, r_y, color=colors['ref'], ls='-', label='Ref.')
            ax.plot(t, traj_y, color=colors['rnn'], ls='--', label='RNN')
            ax.set_ylabel("Y")
            omega = config['f_y'][0] * 2 * np.pi

        ax.set_xlabel("Time")
        ax.set_title(r'$\omega=$' + f'{omega:.2f}')
        if i == 0:
            ax.legend()

    sns.despine()
    plt.tight_layout()
    plt.show()



In [ ]:
# ------------------------------
# Config and Initialization
# ------------------------------
phases_x = np.random.uniform(-np.pi, np.pi, size=len(amps_x))
phases_y = np.random.uniform(-np.pi, np.pi, size=len(amps_y))

# Predefined stimulus configurations for 4 probe conditions
configs = [
    {"a_x": amps_x[:1], "f_x": freqs_x[:1], "a_y": np.zeros(1), "f_y": np.zeros(1)},
    {"a_x": np.zeros(1), "f_x": np.zeros(1), "a_y": amps_y[:1], "f_y": freqs_y[:1]},
    {"a_x": amps_x[1:], "f_x": freqs_x[1:], "a_y": np.zeros(1), "f_y": np.zeros(1)},
    {"a_x": np.zeros(1), "f_x": np.zeros(1), "a_y": amps_y[1:], "f_y": freqs_y[1:]}
]

# Allocate memory to record training stats
loss_matrix = np.zeros((num_epochs, 4))
W_in_all = np.zeros((num_epochs, 4, 100))    # shape: [epoch, input_dim, hidden_dim]
W_out_all = np.zeros((num_epochs, 100, 2))   # shape: [epoch, hidden_dim, output_dim]
W_out_norm = np.zeros((num_epochs,))

# ------------------------------
# Evaluation over Training Epochs
# ------------------------------
for ep in range(num_epochs):

    # Load model weights
    checkpoint = torch.load(f'models/{net_name}/epoch_{ep}.pth')
    rnn.load_state_dict(checkpoint['model_state'])

    # Log input and output weights
    W_in_all[ep] = rnn.Wih.detach().numpy()
    W_out_all[ep] = rnn.Who.detach().numpy()
    W_out_norm[ep] = np.linalg.norm(W_out_all[ep])

    # Loop over each probe config
    for i, config in enumerate(configs):
        # Generate ramped reference trajectory
        r_x = generate_sum_of_sinusoids(t, config["a_x"], config["f_x"], phases_x) * ramp_function(t, 1.0)
        r_y = generate_sum_of_sinusoids(t, config["a_y"], config["f_y"], phases_y) * ramp_function(t, 1.0)
        r_x_torch = torch.tensor(r_x, dtype=torch.float32)
        r_y_torch = torch.tensor(r_y, dtype=torch.float32)

        with torch.no_grad():
            h = torch.zeros((1, rnn.hidden_size), dtype=torch.float32)
            x = torch.zeros(4, dtype=torch.float32)
            episode_loss = []

            for n in range(n_samples):
                input_t = torch.tensor([[x[0], x[2], r_x_torch[n], r_y_torch[n]]], dtype=torch.float32)
                u, h, _ = rnn(input_t, h)
                x = plant.step(x, u[0])

                err = (x[0] - r_x_torch[n])**2 + (x[2] - r_y_torch[n])**2
                episode_loss.append(err.item())

        loss_matrix[ep, i] = np.mean(episode_loss)

# ------------------------------
# Plotting Loss Curves by Frequency
# ------------------------------
omega_labels = [r'$\omega=0.62$', r'$\omega=0.94$', r'$\omega=1.57$', r'$\omega=2.20$']
plt.figure(figsize=(6, 4))

for i in range(4):
    plt.plot(loss_matrix[:, i], label=omega_labels[i])

plt.yscale('log')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Tracking Error for Each Frequency')
plt.legend()
sns.despine()
plt.tight_layout()
plt.show()


In [ ]:
# === Setup ===
dt = 0.1
g = 0.2
num_epochs = 5001
batch_size = 100
save_model = True
net_name = 'open_4_freq_linear_adam'
cutoff_interval = (num_epochs - 1) // 1000
train_losses = []

# === Define Student Network ===
student_net = ct_rnn_controller(
    input_size=4,
    hidden_size=100,
    output_size=2,
    phi='linear',
    manually_initialize=True,
    with_bias=False,
    train_Wih=True,
    train_Whh=True,
    train_Who=True,
    small_scale=False,
    Wih_start_big=False,
    g=g,
    dt=dt,
    tracking_task=True,
)
student_net.load_state_dict(torch.load(f'models/closed_4_freq_linear_adam/epoch_0.pth')['model_state'])

# === Define Teacher Network ===
teacher_net = ct_rnn_controller(
    input_size=4,
    hidden_size=100,
    output_size=2,
    phi='linear',
    manually_initialize=True,
    with_bias=False,
    train_Wih=False,  
    train_Whh=True,
    train_Who=True,
    small_scale=False,
    Wih_start_big=False,
    g=g,
    dt=dt,
    tracking_task=True,
)
teacher_net.load_state_dict(torch.load(f'models/closed_4_freq_linear_adam/epoch_5000.pth')['model_state'])

# === Environment ===
plant = plant_2D_torch(dt=dt, noise=0.0, clamp=True)

# === Optimizer ===
optimizer = optim.Adam([p for p in student_net.parameters() if p.requires_grad], lr=1e-3)
clip_grad = False

# === Create model save directory ===
model_dir = f'models/{net_name}'
if save_model and not os.path.exists(model_dir):
    os.makedirs(model_dir)

# === Training Loop ===
for epoch in range(num_epochs):

    h_student = torch.zeros(batch_size, student_net.hidden_size)
    h_teacher = torch.zeros(batch_size, teacher_net.hidden_size)
    x_current = torch.zeros(batch_size, 4, dtype=torch.float32, requires_grad=True)

    # === Generate sinusoidal reference ===
    phases_x = np.random.uniform(-np.pi, np.pi, size=(batch_size, len(amps_x)))
    phases_y = np.random.uniform(-np.pi, np.pi, size=(batch_size, len(amps_y)))

    r_x = generate_sum_of_sinusoids_batch(t, amps_x, freqs_x, phases_x) * ramp_function(t, ramp_duration=1.0)
    r_y = generate_sum_of_sinusoids_batch(t, amps_y, freqs_y, phases_y) * ramp_function(t, ramp_duration=1.0)

    r_x = torch.tensor(r_x, dtype=torch.float32)
    r_y = torch.tensor(r_y, dtype=torch.float32)


    total_loss = 0.0

    # === Time loop ===
    for i in range(n_samples):
        input_rnn = torch.stack([
            x_current[:, 0],   # x
            x_current[:, 2],   # y
            r_x[:, i],         # x_ref
            r_y[:, i]          # y_ref
        ], dim=-1)

        u_teacher, h_teacher, _ = teacher_net(input_rnn, h_teacher)
        u_student, h_student, _ = student_net(input_rnn, h_student)

        loss_step = torch.mean((u_teacher - u_student) ** 2)
        total_loss += loss_step * dt

        # update environment
        x_next = plant.step(x_current.T, u_teacher.T)
        x_current = x_next.T

    loss = total_loss / n_samples

    # === Backward ===
    if epoch > 0:
        optimizer.zero_grad()
        loss.backward()
        if clip_grad:
            torch.nn.utils.clip_grad_norm_(student_net.parameters(), max_norm=1.0)
        optimizer.step()

    # === Logging ===
    train_losses.append(loss.item())
    if (epoch + 1) % 500 == 0:
        print(f"[Epoch {epoch + 1}/{num_epochs}] Loss = {loss.item():.6f}")

    # === Save model checkpoint ===
    if save_model:
        torch.save({
            'epoch': epoch,
            'model_state': student_net.state_dict(),
            'loss': loss.item()
        }, f'{model_dir}/epoch_{epoch}.pth')


In [ ]:
# === Config ===
net_name = 'open_4_freq_linear_adam'
train_batch_size = 100
test_batch_size = 10
epochs = np.arange(num_epochs)

# === Load training loss from checkpoints ===
train_losses = [
    torch.load(f'models/{net_name}/epoch_{ep}.pth')['loss']
    for ep in epochs
]

# === Evaluate test loss over time ===
test_losses = []

for ep in epochs:
    # Load model
    rnn.load_state_dict(torch.load(f'models/{net_name}/epoch_{ep}.pth')['model_state'])

    # Initialize hidden state and system state
    h = torch.zeros(test_batch_size, rnn.hidden_size, dtype=torch.float32)
    x = torch.zeros(test_batch_size, 4, dtype=torch.float32)  # [x, vx, y, vy]

    # Generate reference trajectories
    phases_x = np.random.uniform(-np.pi, np.pi, size=(test_batch_size, len(amps_x)))
    phases_y = np.random.uniform(-np.pi, np.pi, size=(test_batch_size, len(amps_y)))
    
    r_x = generate_sum_of_sinusoids_batch(t, amps_x, freqs_x, phases_x)
    r_y = generate_sum_of_sinusoids_batch(t, amps_y, freqs_y, phases_y)
    ramp = ramp_function(t, ramp_duration=1.0)

    r_x = torch.tensor(r_x * ramp, dtype=torch.float32)
    r_y = torch.tensor(r_y * ramp, dtype=torch.float32)

    loss = 0.0
    for i in range(n_samples):
        input_t = torch.stack((x[:, 0], x[:, 2], r_x[:, i], r_y[:, i]), dim=1)
        u, h, _ = rnn(input_t, h)

        x_next = plant.step(x.T, u.T)
        err_x = (x_next[0, :] - r_x[:, i]) ** 2
        err_y = (x_next[2, :] - r_y[:, i]) ** 2
        loss += (err_x.mean() + err_y.mean()) * dt
        x = x_next.T

    test_losses.append((loss / n_samples).item())

# === Plot losses ===
fig, ax = plt.subplots(1, 2, figsize=(8, 4))

ax[0].plot(np.log(train_losses), lw=2, color='royalblue')
ax[0].set_title('Train Loss')
ax[0].set_xlabel('Epoch')
ax[0].set_ylabel('Log Loss')

ax[1].plot(np.log(test_losses), lw=2, color='royalblue')
ax[1].set_title('Test Loss')
ax[1].set_xlabel('Epoch')
ax[1].set_ylabel('Log Loss')

sns.despine()
plt.tight_layout()
plt.show()
